# Exploratory Data Analysis of NIH Chest X-ray14 Dataset using Pandas
## Project Overview

This notebook performs exploratory data analysis (EDA) of the NIH Chest X-ray14 dataset using pandas. The primary goal is to reproduce the same analysis previously completed using SQL, allowing a direct comparison between SQL and pandas workflows. This notebook also incorporates data cleaning steps before analysis, following practices in data science. 

## Objectives

- Load and inspect dataset
- Perform basic data cleaning
- Explore patient demographics
- Analyze disease prevalence
- Investigate follow-up information
- Examine multi-label disease findings
- Prepare the dataset for future machine learning projects

## Dataset

- Dataset: NIH ChestX-ray14
- Source: National Institutes of Health (NIH)
- Records: Chest X-ray images with associated patient metadata and radiological findings
___

In [2]:
# Import libraries
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/Users/hweeenheah/Downloads/EDA_Resources/sql-medical-imaging-eda/Data_Entry_2017.csv")
df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


# Data Cleaning

Before performing exploratory data analysis, we inspect the data for common data quality issues such as missing values, duplicate records, and incorrect data types.

In [ ]:
# Dataset Information
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"Number of unique patients: {df['Patient ID'].nunique()}")
print(f"Null values:\n{df.isnull().sum()}")
print(f"Duplicate rows:\n{df.duplicated().sum()}")
print(f"View Position:\n{df['View Position'].value_counts()}")


Rows: 112120, Columns: 12
Number of unique patients: 30805
Null values:
Image Index                         0
Finding Labels                      0
Follow-up #                         0
Patient ID                          0
Patient Age                         0
Patient Gender                      0
View Position                       0
OriginalImage[Width                 0
Height]                             0
OriginalImagePixelSpacing[x         0
y]                                  0
Unnamed: 11                    112120
dtype: int64
Duplicate rows:
0
View Position:
View Position
PA    67310
AP    44810
Name: count, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 112120 entries, 0 to 112119
Data columns (total 12 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Image Index                  112120 non-null  str    
 1   Finding Labels               112120 non-null  str    
 2   Follow-up #          

In [96]:
# Further dataset investigation
# Patient ages > 120, sorted in descending order
df.loc[
    df["Patient Age"] > 120,
    ["Patient ID", "Patient Age"]
].sort_values(["Patient Age"], ascending=False)
# Finding Labels investigation
df["Finding Labels"].nunique()
diseases = (
    df["Finding Labels"]
    .str.split("|")
    .explode()
)
sorted(diseases.unique())
# Follow-up number investigation
df["Follow-up #"].describe()
df["Follow-up #"].value_counts().sort_index().tail(20)


Follow-up #
164    2
165    2
166    2
167    2
168    2
169    2
170    2
171    2
172    2
173    1
174    1
175    1
176    1
177    1
178    1
179    1
180    1
181    1
182    1
183    1
Name: count, dtype: int64

### Dataset Information

The dataset contains 112,120 Chest x-ray records from 30,805 unique patients. Since multiple images exist for many patients, later analyses distinguish between image-level statistics and patient-level statistics.
Dataset did not contain null, missing, or duplicated information. 

During data validation, 16 records were identified with patient ages greater than 120 years (maximum: 414 years), which are not physiologically plausible. No official documentation explaining their encoding was identified, hence will be excluded from analyses involving patient age. The original dataset was otherwise left unchanged to preserve diagnosis and follow-up information.

After splitting the multi-label entries into individual disease labels, no inconsistent spelling, capitalisation differences, or whitespaces were identified. The dataset contains 15 standardised disease labels, including a "No Findings" label.

The follow-up numbers ranges from 0 to 183, with no negative values observed. Median follow-up number was 3, while the average follow-up number was 8.57, indicating that most patients underwent relatively few follow-up investigations, but a small number of patients received substantially more. The gradual decline in frequency in higher follow-up numbers suggest these values likely reflect genuine longitudinal imaging rather than obvious data entry errors.

In [97]:
# Data Cleaning
# df_age to be used for analyses involving age, so that the ages > 120 outliers will be excluded from the analysis, without changing original df
df_age = df[df["Patient Age"] <= 120].copy()

# Exploratory Data Analysis

We will explore, analyse, and query the data in the following categories:
1. Basic shape of data
2. Age distribution by view position
3. Gender distribution
4. Disease prevalence
5. Follow-up patterns
6. Findings by view position
7. Age distribution by disease
8. Findings by follow-up number
9. Disease prevalence per disease
10. Commonly co-occurring diseases

Each query will conclude with a brief explanation detailing its rationale and clinical significance.

In [ ]:
# Query 1: Basic shape of data

print(f"Total records: {df.shape[0]}")
print(f"Number of unique patients: {df['Patient ID'].nunique()}")

Total records: 112120
112120
Number of unique patients: 30805


### Basic shape of data

There are 112,120 total records for 30,805 unique patients. On average, each patient in this cohort underwent an average of 3.6 scans. It shows that the NIH Chest X-ray dataset is an extensive dataset with comprehensive sample size, with meticulous and thorough patient monitoring, ideal for long-term analysis.

In [ ]:
# Query 2: Age distribution of patients by view position
# 16 outlier ages >120 excluded

df_age.groupby('View Position')['Patient Age'].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
View Position,,,,,,,,
AP,44805.0,46.18,17.18,1.0,33.0,48.0,59.0,94.0
PA,67299.0,47.33,16.18,1.0,36.0,49.0,59.0,95.0


### Rationale and Clinical Significance (Age statistics by view position)

After excluding the 16 records with physiologically implausible patient ages (> 120 years old), there are 44,805 Chest X-rays performed in Anteroposterior (AP) view and 67,299 Chest X-rays performed in Posteroanterior (PA) view. The higher number of PA examinations is consistent with standard radiographic practice, where PA is the preferred projection for ambulatory patients. The average patient ages for AP (46.18 years) and PA (47.33 years) are similar, suggesting that patient age alone may not strongly influence projection selection in this dataset.

In [108]:
# Query 3: Gender distribution

df["Patient Gender"].value_counts()
gender_distribution = (df["Patient Gender"].value_counts().to_frame(name="Count"))
gender_distribution["Percentage"] = (gender_distribution["Count"] / len(df) * 100).round(2)
gender_distribution

,Count,Percentage
Patient Gender,,
M,63340,56.49
F,48780,43.51


### Rationale and Clinical Significance (Gender statistics)

Understanding gender distribution of the dataset helps identify potential demographic imbalances, as gender could influence disease prevalence and subsequent model development.

This dataset contains 63,340 (56.49%) male chest x-rays and 48,780 (43.51%) female chest x-rays, demonstrating mildly male-skewed gender distribution. This demographic imbalance should be considered when interpreting disease prevalence in machine learning model developments, as disease patterns may differ between genders, thus affecting model performance.

In [112]:
# Query 4: Disease prevalence

df["Finding Labels"].value_counts().head(15).rename("Occurences").to_frame()

,Occurences
Finding Labels,
No Finding,60361
Infiltration,9547
Atelectasis,4215
Effusion,3955
Nodule,2705
Pneumothorax,2194
Mass,2139
Effusion|Infiltration,1603
Atelectasis|Infiltration,1350


### Rationale and Clinical Significance (Disease Prevalence)

Examining the frequency of radiological findings and their combinations provides an overview of commonly reported radiological findings and how frequently multiple abnormalities co-occur. 
"No Finding" was the most frequently reported label, indicating that a substantial proportion of chest x-rays had no detected abnormalities. Common single-disease findings include infiltration and atelectasis. Multiple-disease findings that also appeared frequently includes effusion and infiltration, or atelectasis and infiltration, reflecting that abnormalities often co-exist in clinical settings.

In [125]:
# Query 5: Follow-up patterns

df.groupby("Patient ID").agg(max_followups=("Follow-up #", 'max'), total_scans=("Follow-up #", 'count')).query("max_followups >10").sort_values("max_followups", ascending=False).head(10)

,max_followups,total_scans
Patient ID,,
10007,183,184
13670,172,173
15530,157,158
12834,156,157
13993,142,143
1836,136,137
19124,129,130
20213,118,119
17138,116,117


### Rationale and Clinical Significance (Follow-ups)

Examining patients with high number of follow-up examinations provides insight into long-term imaging patterns and identifies individuals who required repeated chest x-rays over time. Frequent follow-up imaging may reflect chronic disease monitoring, prolonged hospitalisation, or ongoing assessment of treatment response. 

This dataset includes several patients with more than 10 follow-ups, with the highest follow-up number reaching 183. These findings demonstrate substantial long-term imaging data, making it suitable for studies investigating disease progression and temporal changes in chest x-rays.

In [137]:
# Query 6: Findings by view position

df.groupby(["View Position", "Finding Labels"]).size().reset_index(name="Count").sort_values("Count", ascending=False).head(15)

,View Position,Finding Labels,Count
1206,PA,No Finding,39302
560,AP,No Finding,21059
1170,PA,Infiltration,5270
528,AP,Infiltration,4277
577,PA,Atelectasis,2210
1004,PA,Effusion,2086
0,AP,Atelectasis,2005
1207,PA,Nodule,1924
421,AP,Effusion,1869
1223,PA,Pneumothorax,1506


### Rationale and Clinical Significance (Findings by view position)

Comparing radiographic findings across AP and PA projections helps identify whether certain findings are more commonly associated with a particular imaging position. Because AP examinations are frequently performed on patients who are unable to stand, differences in finding frequencies may reflect differences in patient populations rather than the imaging projection itself.

The most common finding-projection combinations were dominated by "No Finding", followed by several common abnormalities in both AP and PA examinations. Any observed differences should be interpreted cautiously, as projection type is influenced by the patient's clinical condition and mobility rather than causing specific radiographic findings.

In [155]:
# Query 7: Age distribution by disease

df_age.groupby("Finding Labels").agg(Average_age=("Patient Age", "mean"),Scan_count=("Patient Age", "count")).round(2).sort_values("Scan_count", ascending=False).head(15)

,Average_age,Scan_count
Finding Labels,,
No Finding,45.74,60353
Infiltration,45.05,9546
Atelectasis,50.84,4214
Effusion,50.83,3955
Nodule,50.21,2705
Pneumothorax,45.72,2193
Mass,48.22,2139
Effusion|Infiltration,47.60,1603
Atelectasis|Infiltration,48.33,1350


### Rationale and Clinical Significance (Age distribution by disease)

Examining the average patient age for different radiological finding combinations helps identify age-related patterns within the dataset. Certain diseases, such as emphysema and cardiomegaly, may be more frequently observed in older adults while others may occur over a broader age range, making age an important demographic when interpreting imaging findings.

Emphysema has the highest average patient age at 52.36 years old, which is in keeping with it being a chronic disease that develops over years, commonly associated with smoking and ageing. 

Atelectasis, effusion, pleural thickening and their combinations also appear to have an older average patient age, in keep with these findings often occurring in hospitalised, post-operative patients with chronic lung disease.

However, these findings should be interpreted with caution as they contain multiple entries with co-existing findings, and the analysis is based on radiological finding combinations rather than individual diseases.

In [163]:
# Query 8: Findings by follow-up number

#To mirror SQL:
df.groupby(["Follow-up #", "Finding Labels"]).size().reset_index(name="Count").sort_values("Count", ascending=False).head(15)

#To answer clinical question:
(df.groupby("Finding Labels").agg(
        Average_Followup=("Follow-up #", "mean"),
        Maximum_Followup=("Follow-up #", "max"),
        Scan_Count=("Follow-up #", "count"))
    .query("Scan_Count >= 100").round(1).sort_values("Average_Followup", ascending=False).head(15))

,Average_Followup,Maximum_Followup,Scan_Count
Finding Labels,,,
Edema|Infiltration|Pneumonia,25.9,112,137
Infiltration|Pneumothorax,22.9,167,345
Consolidation|Effusion|Infiltration,19.4,108,134
Emphysema|Pneumothorax,19.3,145,337
Atelectasis|Consolidation|Infiltration,19.3,141,104
Consolidation|Infiltration,18.8,118,441
Nodule|Pneumothorax,18.4,109,130
Emphysema|Infiltration,18.3,125,126
Cardiomegaly|Effusion|Infiltration,17.9,87,120


### Rationale and Clinical Significance (Findings by number of follow-ups)

Examining the average follow-up number for radiological finding combinations helps identify which abnormalities tend to appear later in patients' imaging histories. Finding combinations with higher average follow-up numbers may reflect persistent or evolving thoracic conditions that are observed during ongoing clinical monitoring.

To improve the stability of the analysis, only finding combinations with at least 100 scans were included. Several combinations involving Infiltration, Consolidation, Effusion, Edema, and Pneumothorax showed relatively high average follow-up numbers, suggesting that these findings are commonly observed in patients undergoing repeated chest radiography. However, this analysis is descriptive and does not establish that these findings cause more frequent follow-up imaging.

In [167]:
# Query 9: Disease prevalence per disease

split_findings = df["Finding Labels"].str.split("|").explode()
split_df = split_findings.value_counts().reset_index(name="Occurrence")
split_df.head(15)


,Finding Labels,Occurrence
0,No Finding,60361
1,Infiltration,19894
2,Effusion,13317
3,Atelectasis,11559
4,Nodule,6331
5,Mass,5782
6,Pneumothorax,5302
7,Consolidation,4667
8,Pleural_Thickening,3385
9,Cardiomegaly,2776


### Rationale and Clinical Significance (Disease prevalence per disease)

Many chest radiographs contain multiple radiologic findings compressed into a single cell. Treating each finding combination as a unique label underestimates the actual disease prevalence. By splitting multi-label entries into their separate diseases, this analysis gives a better estimate of the true occurence of each radiological finding across the dataset.

After separating each disease from its multi-label entries, Infiltration, Effusion, and Atelectasis were substantially more common than suggested by the earlier combination-based analysis. This marked increase indicates that these findings frequently co-occur with other radiological abnormalities rather than appearing in isolation. These results highlight the importance of appropriate preprocessing for multi-label medical imaging datasets, as analysing only complete finding combinations can substantially underestimate the prevalence of individual diseases.

In [170]:
# Query 10: Which diseases commonly co-occur?


df[df["Finding Labels"].str.contains(r"\|")]["Finding Labels"].value_counts().head(15).rename_axis("Disease Combination").reset_index(name="Occurrence")

,Disease Combination,Occurrence
0,Effusion|Infiltration,1603
1,Atelectasis|Infiltration,1350
2,Atelectasis|Effusion,1165
3,Infiltration|Nodule,829
4,Atelectasis|Effusion|Infiltration,737
5,Cardiomegaly|Effusion,484
6,Consolidation|Infiltration,441
7,Infiltration|Mass,420
8,Effusion|Pneumothorax,403
9,Effusion|Mass,402


### Rationale and Clinical Significance (Commonly co-occurring diseases)

Many chest radiographs contain more than one finding. Identifying the most common disease combination provides insight into commonly co-reported findings and their subsequent diagnoses. 

Most frequent combinations include effusion with infiltration, atelectaasis with infiltration, and atelectasis with effusion. These combinations are clinically plausible as diseases often present with multiple concurrent findings rather than isolated findings. Understanding common co-occurring findings is important in developing multi-label classification models, where several abnormalities may need to be predicted from a single chest x-ray. This can contribute to future models where diagnosis based on commonly co-occurring findings can be trained. For example, effusion with infiltration may be a strong indicator of pneumonia with pleural involvement; atelectasis with infiltration is common is post-operative patients or pleural disease; mass with nodule could indicate multiple pulmonary lesions; cardiomegaly with effusion may occur in patients with heart failure.